# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zubairnajam/Week1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/zubairnajam/FlyRank-AI-ML-_Internship.git

Cloning into 'FlyRank-AI-ML-_Internship'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 126 (delta 39), reused 100 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 1.84 MiB | 13.01 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [4]:
import os
os.chdir("FlyRank-AI-ML-_Internship")
print("Now in:", os.getcwd())
print(os.listdir("data/raw"))

Now in: /content/FlyRank-AI-ML-_Internship
['content_refresh_anonymized.csv']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (content_id), representing its aggregated state over a fixed 90-day trailing window (impressions_90d, sessions_90d, etc. are all 90-day rollups baked into the starter CSV, not daily facts). This is a snapshot grain, not a time-series grain — each content_id appears once, not once per day. If I move to the warehouse later, the grain changes to report_date + client_hash_id + content_hash_id (one row per page per day), which is a different, finer unit of analysis I'd need to re-declare, not assume carries over.

In [5]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"Rows == unique content_id? {len(df) == df['content_id'].nunique()}")

Total rows: 30,000
Unique content_id: 30,000
Rows == unique content_id? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (known before the decision point, safe to use): impressions_90d, sessions_90d, content_age_days, avg_position, ctr, word_count, engagement_rate, scroll_rate, ai_traffic_pct, content-type/intent categoricals.
Label/proxy: trend_direction → derived is_declining_label. Flagged as a proxy, not a true future outcome (per ML-03).
Context (useful to explain a result, not to feed a model): client_id, content_id — join keys and grouping only, never predictive signal themselves.
Excluded, with why: any FlyRank product decision output — health_score, priority_score, action_type, refresh flags. These aren't shipped in the anonymized data at all, but I'm naming them explicitly as excluded-by-design: if I ever rebuild one myself from raw signals, I will not feed it back in as a feature or label, because that would just teach a model to copy an existing rule (circular result), not discover anything.

In [6]:
feature_cols = ["impressions_90d", "sessions_90d", "content_age_days", "avg_position",
                 "ctr", "word_count", "engagement_rate", "scroll_rate"]
label_col = "is_declining_label"
context_cols = ["client_id", "content_id"]
excluded = ["health_score", "priority_score", "action_type"]

present_features = [c for c in feature_cols if c in df.columns]
missing_features = [c for c in feature_cols if c not in df.columns]
print("Present:", present_features)
print("Missing (check column names):", missing_features)
print("Excluded fields present in data?", [c for c in excluded if c in df.columns])

Present: ['impressions_90d', 'sessions_90d', 'content_age_days', 'avg_position', 'ctr', 'word_count', 'engagement_rate', 'scroll_rate']
Missing (check column names): []
Excluded fields present in data? []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim above gets checked here, not assumed.

In [7]:
assert len(df) == df["content_id"].nunique(), "Grain violated: duplicate content_id rows found"

missing_summary = df[present_features + ["client_id", "content_id"]].isna().mean().sort_values(ascending=False)
print(missing_summary)

print("\nMin content_age_days:", df["content_age_days"].min())
print("Rows below 90-day age filter:", (df["content_age_days"] < 90).sum())

print("\nLabel balance:")
print(df["is_declining_label"].value_counts(normalize=True))

print("\nUnique clients:", df["client_id"].nunique())
print(df["client_id"].value_counts().describe())

word_count          0.256633
scroll_rate         0.004167
sessions_90d        0.000000
impressions_90d     0.000000
avg_position        0.000000
content_age_days    0.000000
ctr                 0.000000
engagement_rate     0.000000
client_id           0.000000
content_id          0.000000
dtype: float64

Min content_age_days: 90
Rows below 90-day age filter: 0

Label balance:


KeyError: 'is_declining_label'

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced history: in the warehouse, different clients have very different amounts of tracking history (as little as a few months, up to 12+ for only 9 of 70 clients) — this starter CSV is a fixed 90-day rollup so it hides that unevenness, but I can't assume the same signal density would hold if I move to daily warehouse facts.
GSC-only early rows: in the warehouse, rows before a client's GA4 tracking start date have search data only (ga4_data_available = FALSE) — meaning "no session data" there means "not tracked yet," not "zero traffic." My starter slice doesn't expose this flag, so I have to remember it applies once I touch daily facts, not treat missing sessions as evidence of zero engagement.
Window overlap risk: trend_direction is computed from the same 90-day window as the features that would predict it — meaning right now feature window and label window overlap. This is a leakage risk I inherited from ML-03's proxy-label choice, not something this snapshot alone can fix; a real future-outcome label needs the warehouse's daily facts, split into a prior-90 feature window and a distinct next-30 target window.
No causal claims possible: this data can show that certain signals co-occur with decline, never that fixing something caused recovery — no experiment sits behind any of it.
No raw text: no titles, queries, or URLs are present (by design), so nothing about content quality or semantic relevance can be assessed — only numeric/categorical signals.

In [8]:
overlap_check = df[["trend_direction", "impressions_90d", "sessions_90d"]].head()
print("trend_direction is computed from the same 90d window as these features:")
print(overlap_check)

trend_direction is computed from the same 90d window as these features:
  trend_direction  impressions_90d  sessions_90d
0            down             3803            17
1            down            15320             9
2            down            12581            11
3          stable            11751            78
4            down            19140           145


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.